[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TODO-your-org/BayesFlowTutorial/blob/main/01_Linear_Regression_Solution.ipynb)

<!-- TODO: replace `TODO-your-org/BayesFlowTutorial/blob/main` above with your real GitHub org/repo once this tutorial is pushed, so the Colab badge links to this notebook. -->

# BayesFlow 1 — From Pyro to Amortized Bayesian Inference — Solution

**Difficulty:** 🟢 Beginner (start here)

> ✅ This is the **completed** notebook. Every `TODO` from the exercise is filled in. There is usually more than one good answer — yours may differ and still be correct.

In [ ]:
# ▶️  RUN THIS FIRST.  Installs the packages this notebook needs.
#     On Google Colab this takes ~1 minute. If you already have BayesFlow
#     installed locally, running it again is harmless.
%pip install "bayesflow>=2.0.12" -q
%pip install pyro-ppl -q

In [ ]:
import os
# BayesFlow runs on JAX, PyTorch or TensorFlow. We pick JAX here (installed on Colab).
if not os.environ.get("KERAS_BACKEND"):
    os.environ["KERAS_BACKEND"] = "jax"

> 📚 **Where to read more**
> - BayesFlow website & docs: <https://bayesflow.org>
> - Example gallery (great for copy-paste starting points): <https://bayesflow.org/main/examples.html>
> - API reference: <https://bayesflow.org/main/api/bayesflow.html>
> - Pyro docs (yesterday's tool): <https://pyro.ai> · MCMC/NUTS API: <https://docs.pyro.ai/en/stable/mcmc.html>

## From "one dataset at a time" to "all datasets at once"

Yesterday you used **Pyro** to do Bayesian inference: you wrote a probabilistic model and ran **MCMC (NUTS)** to approximate the posterior $p(\theta \mid x)$ for a dataset. That is exact and general — but it solves **one dataset at a time**. Got 500 datasets? Run NUTS 500 times.

**Amortized Bayesian inference** takes a different route. We train a neural network **once** to map *any* dataset $x$ directly to an approximate posterior $q_\phi(\theta \mid x)$. Training costs more up front, but afterwards inference for a new dataset is a single fast forward pass.

In this notebook we do the same linear-regression problem **both ways** and compare:
1. **Pyro + NUTS** — the familiar route, on one dataset.
2. **BayesFlow** — train an amortized estimator, then reuse it for free.

BayesFlow is organised into a few modules you'll meet below: `simulators` (draw batches), `adapters` (reshape for the networks), `networks` (summary + inference networks), and `workflows` (glue it together and train).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(suppress=True)

## 1. The model — in Pyro

Linear regression:

$$ \beta_0 \sim \mathcal{N}(2, 3), \quad \beta_1 \sim \mathcal{N}(0, 1), \quad \sigma \sim \mathrm{Gamma}(1, 1), \qquad y_i \sim \mathcal{N}(\beta_0 + \beta_1 x_i,\; \sigma). $$

First, let's make **one** observed dataset with known ground-truth parameters (so we can check our answers later).

In [ ]:
true_beta = np.array([2.0, 1.0])   # (intercept, slope)
true_sigma = 0.8
N_obs = 10

rng = np.random.default_rng(7)
x_obs = rng.normal(0, 1, N_obs)
y_obs = rng.normal(true_beta[0] + true_beta[1] * x_obs, true_sigma)

plt.scatter(x_obs, y_obs); plt.xlabel("x"); plt.ylabel("y"); plt.title("Our observed dataset");

Now write the model in Pyro. A Pyro model is a single function with `pyro.sample` statements for the priors and the likelihood — exactly the style from yesterday.

In [ ]:
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import MCMC, NUTS

def pyro_model(x, N, y=None):
    # priors
    beta = pyro.sample("beta", dist.Normal(torch.tensor([2., 0.]),
                                            torch.tensor([3., 1.])).to_event(1))
    sigma = pyro.sample("sigma", dist.Gamma(1., 1.))
    # likelihood
    mean = beta[0] + beta[1] * x
    with pyro.plate("data", N):
        return pyro.sample("obs", dist.Normal(mean, sigma), obs=y)

## 2. Solve it the way you know — Pyro NUTS

Run MCMC on our one dataset to get posterior samples. We time it, because the running time is the whole point of what comes next.

In [ ]:
import time
pyro.set_rng_seed(0)

t0 = time.time()
mcmc = MCMC(NUTS(pyro_model), num_samples=1000, warmup_steps=500)
mcmc.run(torch.tensor(x_obs).float(), N_obs, y=torch.tensor(y_obs).float())
pyro_time = time.time() - t0

pyro_post = mcmc.get_samples()  # {'beta': (1000, 2), 'sigma': (1000,)}
print(f"NUTS finished in {pyro_time:.1f}s for ONE dataset")
print("posterior mean beta:", pyro_post["beta"].mean(0).numpy().round(2),
      " sigma:", float(pyro_post["sigma"].mean()))

> 🔎 That was fast for **one** small dataset. But NUTS starts over for every new dataset, and cost grows with data size and model complexity. If you had 500 datasets to fit, you'd pay that cost 500 times. Enter amortization.

## 3. The same model, as a forward simulator (BayesFlow)

A Pyro model *declares* priors and likelihood together. BayesFlow instead wants a **forward simulator**: functions that **generate** data. It's the same model, just expressed as "draw parameters, then draw data".

- `prior()` → draws $(\beta, \sigma)$ (the same priors as the Pyro model),
- `likelihood(beta, sigma, N)` → draws $(x, y)$,
- `meta()` → draws the dataset size $N$ (so one trained network handles many sizes).

In [ ]:
def likelihood(beta, sigma, N):
    x = np.random.normal(0, 1, size=N)
    y = np.random.normal(beta[0] + beta[1] * x, sigma, size=N)
    return dict(y=y, x=x)

def prior():
    beta = np.random.normal([2, 0], [3, 1])   # matches the Pyro priors
    sigma = np.random.gamma(1, 1)
    return dict(beta=beta, sigma=sigma)

def meta():
    return dict(N=np.random.randint(5, 15))

Combine them into a simulator. We pass `meta` via `meta_fn` so that $N$ is constant **within** each batch (all datasets in a batch must share a shape to become one tensor) but varies **across** batches.

In [ ]:
import bayesflow as bf

simulator = bf.simulators.make_simulator([prior, likelihood], meta_fn=meta)
sim_draws = simulator.sample(500)
print("beta:", sim_draws["beta"].shape, " x:", sim_draws["x"].shape)

### The Adapter

Neural networks want clean, standardized tensors and must be told which variables are **targets** and which are **data**. Key steps: treat the observations as an unordered **set** (regression is order-invariant), keep `sigma` **positive**, and stack variables into the two keys BayesFlow expects: `inference_variables` (targets) and `summary_variables` (data).

In [ ]:
adapter = (
    bf.Adapter()
    .broadcast("N", to="x")
    .as_set(["x", "y"])
    .constrain("sigma", lower=0)
    .sqrt("N")
    .convert_dtype("float64", "float32")
    .concatenate(["beta", "sigma"], into="inference_variables")
    .concatenate(["x", "y"], into="summary_variables")
    .rename("N", "inference_conditions")
)

processed = adapter(sim_draws)
print(processed["summary_variables"].shape, processed["inference_variables"].shape)

### Networks & workflow

A **summary network** compresses a variable-size dataset into a fixed embedding — we use a permutation-invariant `SetTransformer` because our data is a set. An **inference network** (`CouplingFlow`, a normalizing flow) turns that embedding into posterior draws. `summary_dim` should be at least the number of parameters (3).

In [ ]:
summary_network = bf.networks.SetTransformer(summary_dim=10)
inference_network = bf.networks.CouplingFlow(transform="spline")

workflow = bf.BasicWorkflow(
    simulator=simulator, adapter=adapter,
    inference_network=inference_network, summary_network=summary_network,
    standardize=["inference_variables", "summary_variables"],
)

### Train (once!)

We use **online training**: fresh datasets are simulated every batch. This is the up-front cost of amortization — a few minutes on Colab CPU. (On JAX the first steps compile and are slower; that's normal.)

In [ ]:
history = workflow.fit_online(epochs=50, batch_size=64, num_batches_per_epoch=200)
f = bf.diagnostics.plots.loss(history)

### Quick in-silico check: recovery

Before comparing to Pyro, a fast sanity check: on fresh *simulated* datasets (where we know the truth), do posterior estimates track the true values? Points should scatter around the diagonal.

In [ ]:
par_names = [r"$\beta_0$", r"$\beta_1$", r"$\sigma$"]
val_sims = simulator.sample(200)
post_draws = workflow.sample(conditions=val_sims, num_samples=1000)
f = bf.diagnostics.plots.recovery(estimates=post_draws, targets=val_sims, variable_names=par_names)

## 4. Do the two posteriors agree? BayesFlow vs Pyro

The real test: run BayesFlow on the **exact same dataset** we gave to NUTS, and overlay the two posteriors. If BayesFlow learned the right thing, the sample clouds should sit on top of each other.

For inference on a specific dataset we just pass its `x`, `y`, `N` as `conditions` — the adapter applies the same transforms it used in training.

In [ ]:
def compare_posteriors(pyro_post, bf_post, truth):
    names = [r"$\beta_0$", r"$\beta_1$", r"$\sigma$"]
    pyro_arr = np.column_stack([pyro_post["beta"].numpy(), pyro_post["sigma"].numpy()[:, None]])
    bf_arr = np.column_stack([bf_post["beta"][0], bf_post["sigma"][0]])
    fig, ax = plt.subplots(1, 3, figsize=(12, 3), layout="constrained")
    for i, a in enumerate(ax):
        a.hist(pyro_arr[:, i], bins=40, density=True, alpha=0.5, color="#4C72B0", label="Pyro (NUTS)")
        a.hist(bf_arr[:, i], bins=40, density=True, alpha=0.5, color="#E7298A", label="BayesFlow")
        a.axvline(truth[i], color="black", ls="--", lw=1, label="truth")
        a.set_xlabel(names[i])
    ax[0].set_ylabel("density"); ax[0].legend()
    return fig

In [ ]:
obs_dataset = {
    "x": x_obs[None, :].astype("float32"),
    "y": y_obs[None, :].astype("float32"),
    "N": np.array([N_obs]),
}
bf_post = workflow.sample(conditions=obs_dataset, num_samples=1000)

compare_posteriors(pyro_post, bf_post, truth=[true_beta[0], true_beta[1], true_sigma]);

> 🔎 The two posteriors should overlap closely (small differences are expected — NUTS is asymptotically exact, BayesFlow is an amortized *approximation*). Same answer, very different route to get there.

## 5. The payoff: amortization

Here's what the extra training bought us. Pyro solved **one** dataset. BayesFlow can now solve **any number** of datasets, each in a fast forward pass — no retraining, no re-running MCMC.

In [ ]:
import time
many = simulator.sample(200)  # 200 brand-new datasets

t0 = time.time()
post_many = workflow.sample(conditions=many, num_samples=1000)
bf_time = time.time() - t0

print(f"BayesFlow: posteriors for 200 datasets in {bf_time:.1f}s total.")
print(f"Pyro/NUTS: ~{pyro_time:.1f}s PER dataset  ->  ~{pyro_time * 200:.0f}s for 200 datasets"
      " (and that's the easy case).")

> 💡 **This is amortization.** The training cost is paid **once**; inference is then near-free and reusable. When you need posteriors for many datasets — or want instant re-analysis as new data arrives — amortized inference wins. When you only ever have one dataset, classic MCMC like Pyro's NUTS is often the simpler choice. Different tools for different jobs.

## 🎉 Nicely done!

You solved the same problem with **Pyro (NUTS)** and **BayesFlow**, confirmed the posteriors agree, and saw why amortization matters.

**Stretch goals**
1. Give NUTS and BayesFlow a *harder* dataset (larger `N`, or a slope near 0) — do they still agree?
2. Swap the spline coupling flow for `transform="affine"` and re-run the comparison — where does the amortized posterior degrade?
3. How would you *validate* the amortized posterior when you have **no** ground truth and **no** MCMC reference? (That is exactly what **simulation-based calibration** is for — the topic of the next notebook.)

> 📎 **TODO (course):** link yesterday's Pyro materials and the reading on amortized inference / normalizing flows here.